# 07 · Evaluation & validation

**Phase skill: measuring a model on data it never saw, and choosing the right
correlation for the data's scale.**

Notebook 06's R² = 0.82 was scored on the same rows the model was fit on — a rigged
exam. Here you hold data out, look at the residuals, and finish with a comparison of
Pearson vs Spearman on the combat-simulation results.

In [ ]:
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

sys.path.insert(0, str((Path.cwd().parent / "src").resolve()))
from analysis import load_sd_features

DB_PATH = (Path.cwd().parent / "data" / "monsterlab.db").resolve()
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
df = load_sd_features(conn)

FEATURES = ["ac", "best_attack_bonus", "best_avg_damage", "best_num_attacks", "best_stat_mod"]

# Simulation results from the repo's combat simulator (reports/sim_results.csv):
# one row per monster -- printed level, and the party's win_rate from 2000 simulated fights.
sim = pd.read_csv(Path.cwd().parent / "reports" / "sim_results.csv")
sim_matched = sim.loc[sim["level"] <= 10].copy()
print(f"{len(df)} monsters, {len(sim)} simulated matchups ({len(sim_matched)} at LV <= 10)")

## Exercise 7.1 — hold data out

Split the data, fit on the training portion only, and score both portions.
Use the five `FEATURES` (NaNs filled with 0), target `level`, **25% test size,
`random_state=42`** so the check can reproduce your split.

**Produce:** `X_train, X_test, y_train, y_test`, a model fit **only on the training
rows**, and `train_r2`, `test_r2`.

<details><summary>Hint 1 (concept)</summary>

The test score answers the only question that matters: how wrong will this model be on a monster it has never met? That requires rows the fit never touched.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up `sklearn.model_selection.train_test_split`.

</details>

In [ ]:
# Produce: X_train, X_test, y_train, y_test, a model fit on train only, train_r2, test_r2

X_train, X_test, y_train, y_test = ...
model = ...
train_r2 = ...
test_r2 = ...

print(f"train R2 = {train_r2:.3f}   test R2 = {test_r2:.3f}")

In [ ]:
assert len(X_train) == 182 and len(X_test) == 61, (
    f"split sizes are {len(X_train)}/{len(X_test)}, expected 182/61 -- test_size=0.25, random_state=42")
assert abs(train_r2 - 0.820) < 0.01, f"train_r2 = {train_r2:.3f}, expected about 0.820"
assert abs(test_r2 - 0.804) < 0.01, f"test_r2 = {test_r2:.3f}, expected about 0.804"
print(f"PASS: train R2 = {train_r2:.3f}, test R2 = {test_r2:.3f}.")
print("Context: a two-point drop from train to test is healthy -- the model generalizes instead of memorizing.")

## Exercise 7.2 — look at the residuals

Scores summarize; residuals confess. Compute the **test-set** residuals
(`y_test − predictions`) and scatter them against the predicted values, with a
horizontal line at 0.

**Produce:** the plot, and `test_residuals` — the residual Series/array for the 61 test
rows.

<details><summary>Hint 1 (concept)</summary>

A good residual plot is boring: a shapeless cloud around zero. Any structure -- a funnel, a curve, a drifting mean -- is the model telling you what it is missing.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up `model.predict` and `ax.axhline`.

</details>

In [ ]:
# Produce: test_residuals (length 61) and its scatter against predictions

test_residuals = ...

print(f"mean residual = {float(pd.Series(test_residuals).mean()):.3f}")

In [ ]:
assert len(test_residuals) == 61, f"test_residuals has {len(test_residuals)} values, expected the 61 test rows"
mean_res = float(pd.Series(list(test_residuals)).mean())
assert abs(mean_res) < 0.5, f"mean test residual = {mean_res:.3f} -- residuals should center near 0 (check the subtraction order)"
print(f"PASS: 61 test residuals centered at {mean_res:+.2f}.")
print("Context: the sign convention matters -- positive residual = the printed LV is higher than the stats predict.")

**State in one sentence what your residual plot claims** (shape, spread, any pattern):

*…*

## Exercise 7.3 — Pearson vs Spearman on the simulation

`sim` holds one row per monster: its printed `level` and the party's `win_rate` over
2000 simulated fights. The reference party scales with monster level only through LV 10;
above that it is clamped at level 10. These checks therefore use the matched LV ≤ 10 set,
where the comparison isolates the monster's simulated difficulty. How well does printed
LV track that difficulty?

Compute both correlations between `win_rate` and `level` in `sim_matched`:

**Produce:** `pearson_r` and `spearman_r` (floats — just the statistic, not the
p-value).

<details><summary>Hint 1 (concept)</summary>

Pearson asks: do the values move together along a line? Spearman asks only: do the rankings agree? When one variable is ordinal-ish, ranks are the safer claim.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up `scipy.stats.pearsonr` and `scipy.stats.spearmanr` -- both return more than one value.

</details>

In [ ]:
# Produce: pearson_r (float), spearman_r (float) -- sim_matched win_rate vs level

pearson_r = ...
spearman_r = ...

print(f"Pearson r = {pearson_r:.3f}   Spearman rho = {spearman_r:.3f}")

In [ ]:
assert abs(pearson_r - (-0.4098)) < 0.005, (
    f"pearson_r = {pearson_r:.4f}, expected about -0.410 -- matched-party win_rate against level")
assert abs(spearman_r - (-0.4083)) < 0.005, (
    f"spearman_r = {spearman_r:.4f}, expected about -0.408")
print(f"PASS: Pearson {pearson_r:.3f}, Spearman {spearman_r:.3f}.")
print("Context: both are moderately negative: within the matched LV range, higher-LV monsters "
      "generally win more often against their reference parties, but neither metric is a complete difficulty measure.")

**Writing prompt.** Monster LV is not a measurement; it is a *label on an ordered
ladder* — the gap between LV 1 and 2 need not equal the gap between LV 11 and 12. Two
or three sentences: why does that make rank correlation (Spearman) the more defensible
summary here, and what extra claim would you be smuggling in by quoting Pearson?

*…*

That question — *which claims survive scrutiny* — is exactly what `08_communicate.ipynb`
is about.